# 03 - SQL Analysis & Relational Schema



In [1]:
import os
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, round as sround


In [2]:
spark = (
    SparkSession.builder
    .appName("Bus_SQL_Integration")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.local.ip", "127.0.0.1")
    .config("spark.driver.memory", "4g")
    .config("spark.python.worker.reuse", "true")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.default.parallelism", "16")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("Cores available to Spark:", spark.sparkContext.defaultParallelism)


Cores available to Spark: 16


In [3]:
project_path = r"D:\BigDataCoursework"
processed_path = os.path.join(project_path, "Data", "processed")

journeys_df = spark.read.parquet(os.path.join(processed_path, "bus_risk_dataset.parquet"))
disruption_df = spark.read.parquet(os.path.join(processed_path, "disruptions.parquet"))
vehicle_df    = spark.read.parquet(os.path.join(processed_path, "vehicle_locations.parquet"))

print("Timetable (journeys) rows:", journeys_df.count())
print("Disruption rows:", disruption_df.count())
print("Vehicle location rows:", vehicle_df.count())


Timetable (journeys) rows: 227010
Disruption rows: 1530
Vehicle location rows: 27987


In [4]:
timing_df = spark.read.parquet(os.path.join(processed_path, "timing_links.parquet"))
timing_df = timing_df.repartition(max(4, spark.sparkContext.defaultParallelism)).cache()
print("Timing link rows:", timing_df.count())
print("Partitions - timing links:", timing_df.rdd.getNumPartitions())


Timing link rows: 1825566
Partitions - timing links: 16


In [5]:
print("Partitions - journeys:", journeys_df.rdd.getNumPartitions())
print("Partitions - disruptions:", disruption_df.rdd.getNumPartitions())
print("Partitions - vehicle:", vehicle_df.rdd.getNumPartitions())

journeys_df = journeys_df.repartition(max(4, spark.sparkContext.defaultParallelism)).cache()
disruption_df = disruption_df.repartition(max(4, spark.sparkContext.defaultParallelism)).cache()
vehicle_df = vehicle_df.repartition(max(4, spark.sparkContext.defaultParallelism)).cache()

journeys_df.count(); disruption_df.count(); vehicle_df.count()  # materialise caches
print("Repartitioned - journeys:", journeys_df.rdd.getNumPartitions())


Partitions - journeys: 16
Partitions - disruptions: 16
Partitions - vehicle: 16
Repartitioned - journeys: 16


In [6]:
journeys_df.createOrReplaceTempView("timetable_journeys")
disruption_df.createOrReplaceTempView("disruptions")
vehicle_df.createOrReplaceTempView("vehicle_positions")

print("3 separate temp views created: timetable_journeys, disruptions, vehicle_positions")


3 separate temp views created: timetable_journeys, disruptions, vehicle_positions


## Cross-dataset joins (Timetable x Disruptions x Vehicle Location)

In [7]:
vehicle_summary_sql = spark.sql("""
    SELECT Operator_ID, Line_Ref, COUNT(*) AS Position_Reports
    FROM vehicle_positions
    GROUP BY Operator_ID, Line_Ref
""")
vehicle_summary_sql.show(10, truncate=False)


+-----------+--------+----------------+
|Operator_ID|Line_Ref|Position_Reports|
+-----------+--------+----------------+
|TFLO       |24      |14              |
|TLCT       |678     |1               |
|TFLO       |141     |9               |
|TBTN       |3C      |7               |
|FBRI       |76      |14              |
|TFLO       |90      |10              |
|ZSIN       |606     |1               |
|SCNH       |4       |11              |
|TANV       |81      |2               |
|RBUS       |15      |2               |
+-----------+--------+----------------+
only showing top 10 rows



In [8]:
integrated_journeys = spark.sql("""
    SELECT
        t.Journey_ID,
        t.Operator_ID,
        t.Service_Code,
        t.Line_Name,
        t.Departure_Hour,
        t.Peak_Hour,
        t.Route_Complexity,
        t.Number_of_Stops,
        t.Risk_Score,
        t.Service_Risk,
        COALESCE(d.Disruption_Count, 0)          AS Disruption_Count,
        CASE WHEN d.Disruption_Count > 0 THEN 1 ELSE 0 END AS Has_Disruption,
        COALESCE(v.Position_Reports, 0)          AS Position_Reports,
        CASE WHEN v.Position_Reports > 0 THEN 1 ELSE 0 END AS Has_Live_Tracking
    FROM timetable_journeys t
    LEFT JOIN (
        SELECT Operator_ID, COUNT(*) AS Disruption_Count
        FROM disruptions
        GROUP BY Operator_ID
    ) d ON t.Operator_ID = d.Operator_ID
    LEFT JOIN (
        SELECT Operator_ID, COUNT(*) AS Position_Reports
        FROM vehicle_positions
        GROUP BY Operator_ID
    ) v ON t.Operator_ID = v.Operator_ID
""")

integrated_journeys.createOrReplaceTempView("integrated_journeys")
print("Integrated rows (timetable + disruptions + vehicle location joined):", integrated_journeys.count())
integrated_journeys.show(10, truncate=False)


Integrated rows (timetable + disruptions + vehicle location joined): 227010
+----------+-----------+------------+----------------------+--------------+---------+----------------+---------------+------------------+------------+----------------+--------------+----------------+-----------------+
|Journey_ID|Operator_ID|Service_Code|Line_Name             |Departure_Hour|Peak_Hour|Route_Complexity|Number_of_Stops|Risk_Score        |Service_Risk|Disruption_Count|Has_Disruption|Position_Reports|Has_Live_Tracking|
+----------+-----------+------------+----------------------+--------------+---------+----------------+---------------+------------------+------------+----------------+--------------+----------------+-----------------+
|VJ166     |PERY       |PB1082425:1 |PERY:PB1082425:1:67   |7             |1        |290             |38             |39.629999999999995|1           |0               |0             |10              |1                |
|VJ1071    |Unknown    |SER5        |SL4            

## Analytical SQL queries 

In [9]:
risk_summary = spark.sql("""
    SELECT Service_Risk, COUNT(*) AS Number_of_Services
    FROM integrated_journeys
    GROUP BY Service_Risk
    ORDER BY Service_Risk
""")
risk_summary.show()


+------------+------------------+
|Service_Risk|Number_of_Services|
+------------+------------------+
|           0|             54242|
|           1|            101722|
|           2|             71046|
+------------+------------------+



In [10]:
disruption_impact = spark.sql("""
    SELECT
        Has_Disruption,
        ROUND(AVG(Risk_Score), 2) AS Avg_Risk_Score,
        COUNT(*) AS Total_Journeys
    FROM integrated_journeys
    GROUP BY Has_Disruption
""")
disruption_impact.show()


+--------------+--------------+--------------+
|Has_Disruption|Avg_Risk_Score|Total_Journeys|
+--------------+--------------+--------------+
|             1|         46.03|         57125|
|             0|         80.87|        169885|
+--------------+--------------+--------------+



In [11]:
operator_performance = spark.sql("""
    SELECT
        Operator_ID,
        COUNT(DISTINCT Journey_ID) AS Total_Journeys,
        ROUND(AVG(Route_Complexity), 2) AS Avg_Complexity,
        SUM(Disruption_Count) AS Total_Disruptions,
        SUM(Has_Live_Tracking) AS Journeys_With_Tracking
    FROM integrated_journeys
    GROUP BY Operator_ID
    ORDER BY Total_Disruptions DESC
    LIMIT 10
""")
operator_performance.show()


+-----------+--------------+--------------+-----------------+----------------------+
|Operator_ID|Total_Journeys|Avg_Complexity|Total_Disruptions|Journeys_With_Tracking|
+-----------+--------------+--------------+-----------------+----------------------+
|    Unknown|          1689|        7512.0|          2642976|                     0|
|       KEMT|            22|        102.24|            10416|                  1736|
|       DGTR|            91|         11.23|              327|                   327|
|       MRDL|            16|         39.18|                0|                     0|
|       KEVE|             4|         20.71|                0|                    35|
|       NUTT|            10|          36.0|                0|                    19|
|       FOWT|             4|         128.0|                0|                    81|
|       EVGC|             6|          10.0|                0|                     0|
|       PCCO|            86|         436.4|                0|    

In [12]:
peak_hour_analysis = spark.sql("""
    SELECT Peak_Hour, Service_Risk, COUNT(*) AS Total
    FROM integrated_journeys
    GROUP BY Peak_Hour, Service_Risk
    ORDER BY Peak_Hour, Service_Risk
""")
peak_hour_analysis.show()


+---------+------------+-----+
|Peak_Hour|Service_Risk|Total|
+---------+------------+-----+
|        0|           0|44717|
|        0|           1|43386|
|        0|           2|33259|
|        1|           0| 9525|
|        1|           1|58336|
|        1|           2|37787|
+---------+------------+-----+



In [13]:
integrated_journeys.write.mode("overwrite").parquet(
    os.path.join(processed_path, "integrated_journeys.parquet")
)

integrated_journeys.coalesce(1).write.mode("overwrite").option("header", True).csv(
    os.path.join(processed_path, "integrated_journeys_csv")
)

print("Integrated dataset exported (Parquet + CSV).")


Integrated dataset exported (Parquet + CSV).


In [16]:
dim_operator_pd = operator_performance.toPandas()
dim_operator_pd = dim_operator_pd.rename(columns={"Avg_Complexity": "Average_Route_Complexity"})


all_operator_ids = (
    set(dim_operator_pd["Operator_ID"])
    | set(disruption_df.select("Operator_ID").distinct().toPandas()["Operator_ID"])
    | set(vehicle_summary_sql.select("Operator_ID").distinct().toPandas()["Operator_ID"])
)
missing_operator_ids = all_operator_ids - set(dim_operator_pd["Operator_ID"])

if missing_operator_ids:
    filler = pd.DataFrame({
        "Operator_ID": sorted(missing_operator_ids),
        "Total_Journeys": 0,
        "Average_Route_Complexity": 0.0,
        "Total_Disruptions": 0,
        "Journeys_With_Tracking": 0,
    })
    dim_operator_pd = pd.concat([dim_operator_pd, filler], ignore_index=True)

print("dim_operator rows after adding disruption/vehicle-only operators:", len(dim_operator_pd))
print("Added", len(missing_operator_ids), "operator(s) not present in the timetable catalogue.")

dim_service_cols = ["Service_Code", "Line_Name"] + (
    ["JourneyPatternRef"] if "JourneyPatternRef" in journeys_df.columns else []
)
dim_service_pd = journeys_df.select(*dim_service_cols).dropDuplicates(["Service_Code"]).toPandas()

if "JourneyPatternRef" not in dim_service_pd.columns:
    dim_service_pd["JourneyPatternRef"] = dim_service_pd["Service_Code"]


fact_journey_spark = integrated_journeys.dropDuplicates(["Journey_ID", "Operator_ID"])
print("fact_journey rows after de-duplication to journey grain:", fact_journey_spark.count())
fact_journey_pd = fact_journey_spark.toPandas()
fact_disruption_pd = disruption_df.toPandas()


for col in ["Start_Time", "End_Time"]:
    fact_disruption_pd[col] = (
        pd.to_datetime(fact_disruption_pd[col], errors="coerce", utc=True)
        .dt.strftime("%Y-%m-%d %H:%M:%S")
    )
fact_vehicle_pd = vehicle_summary_sql.toPandas()

print("dim_operator:", len(dim_operator_pd))
print("dim_service:", len(dim_service_pd))
print("fact_journey:", len(fact_journey_pd))
print("fact_disruption:", len(fact_disruption_pd))
print("fact_vehicle_position:", len(fact_vehicle_pd))


dim_operator rows after adding disruption/vehicle-only operators: 377
Added 367 operator(s) not present in the timetable catalogue.
fact_journey rows after de-duplication to journey grain: 3086
dim_operator: 377
dim_service: 304
fact_journey: 3086
fact_disruption: 1530
fact_vehicle_position: 7042


In [17]:
mysql_import_path = os.path.join(processed_path, "mysql_import")
os.makedirs(mysql_import_path, exist_ok=True)

dim_operator_pd[["Operator_ID", "Total_Journeys", "Average_Route_Complexity",
                  "Total_Disruptions", "Journeys_With_Tracking"]].to_csv(
    os.path.join(mysql_import_path, "dim_operator.csv"), index=False, na_rep=r"\N")

dim_service_pd[["Service_Code", "Line_Name", "JourneyPatternRef"]].to_csv(
    os.path.join(mysql_import_path, "dim_service.csv"), index=False, na_rep=r"\N")

fact_journey_pd[["Journey_ID", "Operator_ID", "Service_Code", "Departure_Hour", "Peak_Hour",
                  "Route_Complexity", "Number_of_Stops", "Risk_Score", "Service_Risk",
                  "Disruption_Count", "Has_Disruption", "Position_Reports",
                  "Has_Live_Tracking"]].to_csv(
    os.path.join(mysql_import_path, "fact_journey.csv"), index=False, na_rep=r"\N")


fact_disruption_pd[["Situation_ID", "Operator_ID", "Line_Ref", "Severity",
                     "Reason", "Start_Time", "End_Time"]].to_csv(
    os.path.join(mysql_import_path, "fact_disruption.csv"), index=False, na_rep=r"\N")

fact_vehicle_pd[["Operator_ID", "Line_Ref", "Position_Reports"]].to_csv(
    os.path.join(mysql_import_path, "fact_vehicle_position.csv"), index=False, na_rep=r"\N")

(
    timing_df
    .select("Journey_ID", "Operator_ID", "From_Stop", "To_Stop", "Run_Time")
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(os.path.join(mysql_import_path, "fact_timing_link_csv"))
)

print("CSV files written to:", mysql_import_path)
print("Import each into MySQL Workbench via Table Data Import Wizard after running mysql_schema.sql")


CSV files written to: D:\BigDataCoursework\Data\processed\mysql_import
Import each into MySQL Workbench via Table Data Import Wizard after running mysql_schema.sql


In [18]:
schema_sql = """
CREATE DATABASE IF NOT EXISTS bus_analytics;
USE bus_analytics;

DROP TABLE IF EXISTS fact_timing_link;
DROP TABLE IF EXISTS fact_vehicle_position;
DROP TABLE IF EXISTS fact_disruption;
DROP TABLE IF EXISTS fact_journey;
DROP TABLE IF EXISTS dim_service;
DROP TABLE IF EXISTS dim_operator;

CREATE TABLE dim_operator (
    Operator_ID              VARCHAR(50) PRIMARY KEY,
    Total_Journeys           INT,
    Average_Route_Complexity DOUBLE,
    Total_Disruptions        INT,
    Journeys_With_Tracking   INT
) ENGINE=InnoDB;

CREATE TABLE dim_service (
    Service_Code       VARCHAR(50) PRIMARY KEY,
    Line_Name          VARCHAR(100),
    JourneyPatternRef  VARCHAR(100)
) ENGINE=InnoDB;

CREATE TABLE fact_journey (
    Journey_ID          VARCHAR(50),
    Operator_ID         VARCHAR(50),
    Service_Code        VARCHAR(50),
    Departure_Hour      INT,
    Peak_Hour           TINYINT,
    Route_Complexity    INT,
    Number_of_Stops     INT,
    Risk_Score          DOUBLE,
    Service_Risk        INT,
    Disruption_Count    INT,
    Has_Disruption      TINYINT,
    Position_Reports    INT,
    Has_Live_Tracking   TINYINT,
    PRIMARY KEY (Journey_ID, Operator_ID),
    FOREIGN KEY (Operator_ID) REFERENCES dim_operator(Operator_ID),
    FOREIGN KEY (Service_Code) REFERENCES dim_service(Service_Code)
) ENGINE=InnoDB;

CREATE TABLE fact_disruption (
    Situation_ID  VARCHAR(50),
    Operator_ID   VARCHAR(50),
    Line_Ref      VARCHAR(50),
    Severity      VARCHAR(30),
    Reason        VARCHAR(100),
    Start_Time    DATETIME NULL,
    End_Time      DATETIME NULL,
    FOREIGN KEY (Operator_ID) REFERENCES dim_operator(Operator_ID)
) ENGINE=InnoDB;

CREATE TABLE fact_vehicle_position (
    Operator_ID       VARCHAR(50),
    Line_Ref          VARCHAR(50),
    Position_Reports  INT,
    FOREIGN KEY (Operator_ID) REFERENCES dim_operator(Operator_ID)
) ENGINE=InnoDB;

CREATE TABLE fact_timing_link (
    Link_ID       BIGINT AUTO_INCREMENT PRIMARY KEY,
    Journey_ID    VARCHAR(50),
    Operator_ID   VARCHAR(50),
    From_Stop     VARCHAR(50),
    To_Stop       VARCHAR(50),
    Run_Time      VARCHAR(20),
    FOREIGN KEY (Journey_ID, Operator_ID) REFERENCES fact_journey(Journey_ID, Operator_ID)
) ENGINE=InnoDB;
"""

schema_path = os.path.join(mysql_import_path, "mysql_schema.sql")
with open(schema_path, "w") as f:
    f.write(schema_sql)

print("Schema written to:", schema_path)
print("Open this file in MySQL Workbench and run it (lightning-bolt icon) before importing the CSVs.")


Schema written to: D:\BigDataCoursework\Data\processed\mysql_import\mysql_schema.sql
Open this file in MySQL Workbench and run it (lightning-bolt icon) before importing the CSVs.


In [19]:

sample_queries_sql = """
USE bus_analytics;

-- 1. Plain analytical queries (safe: no variable user input, so no injection risk regardless)
SELECT Service_Risk, COUNT(*) AS Number_of_Services
FROM fact_journey
GROUP BY Service_Risk
ORDER BY Service_Risk;

SELECT o.Operator_ID, o.Total_Journeys, o.Total_Disruptions, o.Journeys_With_Tracking
FROM dim_operator o
ORDER BY o.Total_Disruptions DESC
LIMIT 10;

SELECT d.Operator_ID, COUNT(*) AS Disruption_Count, d.Severity
FROM fact_disruption d
GROUP BY d.Operator_ID, d.Severity
ORDER BY Disruption_Count DESC
LIMIT 10;

-- 2. Parameterised query using MySQL's native PREPARE/EXECUTE - the placeholder (?) is bound
--    at execution time rather than concatenated into the SQL string, which is exactly what
--    "parameterised queries to prevent SQL injection" means, without needing a Python connector.
PREPARE journey_lookup FROM
    'SELECT * FROM fact_journey WHERE Operator_ID = ? LIMIT 5';

SET @operator_param = 'TFLO';   -- change this to any real Operator_ID
EXECUTE journey_lookup USING @operator_param;

DEALLOCATE PREPARE journey_lookup;
"""

queries_path = os.path.join(mysql_import_path, "sample_queries.sql")
with open(queries_path, "w") as f:
    f.write(sample_queries_sql)

print("Sample queries (incl. parameterised PREPARE/EXECUTE) written to:", queries_path)
print("Run these in MySQL Workbench's Query tab and screenshot the results for the report appendix.")


Sample queries (incl. parameterised PREPARE/EXECUTE) written to: D:\BigDataCoursework\Data\processed\mysql_import\sample_queries.sql
Run these in MySQL Workbench's Query tab and screenshot the results for the report appendix.


In [20]:
journeys_df.unpersist(); disruption_df.unpersist(); vehicle_df.unpersist()


DataFrame[Journey_ID: string, Operator_ID: string, Line_Ref: string, Line_Name: string, Direction: string, Origin: string, Destination: string, Latitude: string, Longitude: string, Recorded_At: string]

In [21]:
spark.stop()
print("Spark stopped successfully.")


Spark stopped successfully.
